# IssuesEvent pipeline test (`03_issue_events`)

This notebook tests `03_issue_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_issue`. It runs the real notebook with `%run` and `test_schema = test_issue`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `03_issue_events` (they create `silver.issue_events` and `quarantine.quarantine_issues_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to IssuesEvent: the five issue fields are read and converted; a missing `issue_closed_at` is normal for an open issue but warned on a `closed` action; a close time before the creation time is warned and only `issue_closed_at` is set to `NULL`; a missing creation time is never filled from the event time; missing issue IDs are kept with warnings; an unexpected action is kept with an `UNEXPECTED_VALUE` warning; and the quarantine table keeps the issue fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_issue"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("issue_events")
TEST_QUARANTINE = T("quarantine_issues_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.issue_events",
    "github_lakehouse.quarantine.quarantine_issues_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 03_issue_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.issue_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_issues_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default issue is `id 7001`, `number 11`, created `2024-01-15T08:00:00Z`, `closed_at = null`, action `opened`.

| Row | Case | Expected result |
|---|---|---|
| `i1` | clean `opened` issue | Silver, every field converted, no warning |
| `i2` | clean `closed` issue with a close time | Silver, no warning |
| `i3` | `closed` action, `closed_at` missing | Silver, warning `MISSING: issue_closed_at` |
| `i4` | `opened`, `closed_at = "not-a-date"` | Silver, NULL, warning `INVALID_CAST: issue_closed_at` |
| `i5` | `opened`, `closed_at` blank | Silver, NULL, no warning (an open issue has no close time) |
| `i6` | `closed`, closed 2024-01-01, created 2024-02-01 | Silver, `issue_closed_at` NULL, `issue_created_at` kept, warning `CLOSED_BEFORE_CREATED: issue_closed_at` |
| `i7` | `issue.id` missing | Silver, `issue_number` kept, warning `MISSING: issue_id` |
| `i8` | no `issue` object | Silver, three `MISSING` warnings; `issue_created_at` stays NULL |
| `i9` | `action = labeled` | Silver, value kept, warning `UNEXPECTED_VALUE: issue_action` |
| `i10` | `action` missing | Silver, warning `MISSING: issue_action` only |
| `i11` | `closed`, `created_at = "bad"`, valid close time | Silver, warning `INVALID_CAST: issue_created_at`, close time kept |
| `i12` | `issue` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `i13` | `repo.id` missing | quarantine `MISSING: repo_id`, issue fields and original text kept |
| `i14` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `x1` | an IssueCommentEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

def issue(drop=(), **changes):
    """An issue object. Defaults = an open issue."""
    i = {"id": 7001, "number": 11, "created_at": "2024-01-15T08:00:00Z", "closed_at": None}
    i.update(changes)
    for key in drop:
        i.pop(key)
    return i

def ev(event_id, type_="IssuesEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, action="opened", iss=None, payload=None):
    """One GitHub-style event as a JSON string. Defaults = a clean IssuesEvent."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": ({"action": action, "issue": issue() if iss is None else iss}
                    if payload is None else payload),
    }
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                     # case
    (ev("i1"),                                                   F1, D1, I1),   # clean, opened
    (ev("i2", action="closed",
        iss=issue(closed_at="2024-02-01T09:00:00Z")),            F1, D1, I1),   # clean, closed
    (ev("i3", action="closed"),                                  F1, D1, I1),   # closed, no close time
    (ev("i4", iss=issue(closed_at="not-a-date")),                F1, D1, I1),   # bad close time
    (ev("i5", iss=issue(closed_at="  ")),                        F1, D1, I1),   # blank close time, open
    (ev("i6", action="closed",
        iss=issue(created_at="2024-02-01T09:00:00Z",
                  closed_at="2024-01-01T00:00:00Z")),            F1, D1, I1),   # closed before created
    (ev("i7", iss=issue(drop=["id"])),                           F1, D1, I1),   # issue_id missing
    (ev("i8", payload={"action": "opened"}),                     F1, D1, I1),   # no issue object
    (ev("i9", action="labeled"),                                 F1, D1, I1),   # unexpected action
    (ev("i10", payload={"issue": issue()}),                      F1, D1, I1),   # action missing
    (ev("i11", action="closed",
        iss=issue(created_at="bad",
                  closed_at="2024-02-01T09:00:00Z")),            F1, D1, I1),   # bad creation time
    (ev("i12", iss="oops"),                                      F1, D1, I1),   # parse error
    (ev("i13", repo={"name": "octo/repo"}),                      F1, D1, I1),   # repo_id missing
    (ev("i14", iss=issue(id=1401)),                              F1, D1, I1),   # duplicate, older
    (ev("i14", iss=issue(id=1402)),                              F2, D1, I2),   # duplicate, newer -> wins
    (ev("x1", type_="IssueCommentEvent"),                        F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 16)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../03_issue_events $test_schema="test_issue" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_issue_action": 1, "null_issue_id": 2, "null_issue_number": 1,
    "null_issue_id_and_number": 1, "null_issue_created_at": 2,
    "closed_without_issue_closed_at": 1, "unexpected_issue_action": 1,
    "issue_closed_before_created": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_issue_events")
check("audit event_type", audit["event_type"], "IssuesEvent")

# --- audit: 15 IssuesEvent rows (x1 is an IssueCommentEvent)
check_audit({
    "bronze_input_rows": 15, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 13,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 12,
    "silver_rows_inserted": 12, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, issue fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "issue_action", "issue_id", "issue_number", "issue_created_at", "issue_closed_at",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      sorted(["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8", "i9", "i10", "i11", "i14"]))

def times(event_id):
    """(issue_created_at, issue_closed_at) as UTC text, or None."""
    found = (spark.table(TEST_SILVER).where(f"event_id = '{event_id}'")
             .selectExpr("date_format(issue_created_at, 'yyyy-MM-dd HH:mm:ss') AS c",
                         "date_format(issue_closed_at, 'yyyy-MM-dd HH:mm:ss') AS x")
             .collect())
    return (found[0]["c"], found[0]["x"]) if found else "row not found"

r = one_silver("i1")
check("i1 issue fields", (r.get("issue_action"), r.get("issue_id"), r.get("issue_number")),
      ("opened", 7001, 11))
check("i1 times (UTC)", times("i1"), ("2024-01-15 08:00:00", None))
check("i1 warning", r.get("quarantine_warning"), None)

r = one_silver("i2")
check("i2 times", times("i2"), ("2024-01-15 08:00:00", "2024-02-01 09:00:00"))
check("i2 warning", r.get("quarantine_warning"), None)

r = one_silver("i3")
check("i3 warning", r.get("quarantine_warning"), "MISSING: issue_closed_at")

r = one_silver("i4")
check("i4 closed_at", r.get("issue_closed_at"), None)
check("i4 warning", r.get("quarantine_warning"), "INVALID_CAST: issue_closed_at")

r = one_silver("i5")
check("i5 closed_at", r.get("issue_closed_at"), None)
check("i5 no warning (open issue)", r.get("quarantine_warning"), None)

r = one_silver("i6")
check("i6 only closed_at cleaned", times("i6"), ("2024-02-01 09:00:00", None))
check("i6 action kept", r.get("issue_action"), "closed")
check("i6 warning", r.get("quarantine_warning"), "CLOSED_BEFORE_CREATED: issue_closed_at")

r = one_silver("i7")
check("i7 ids", (r.get("issue_id"), r.get("issue_number")), (None, 11))
check("i7 warning", r.get("quarantine_warning"), "MISSING: issue_id")

r = one_silver("i8")
check("i8 issue fields", (r.get("issue_id"), r.get("issue_number"), r.get("issue_created_at")),
      (None, None, None))
check("i8 event_timestamp still set", r.get("event_timestamp") is not None, True)
check("i8 warnings", r.get("quarantine_warning"),
      "MISSING: issue_id; MISSING: issue_number; MISSING: issue_created_at")

r = one_silver("i9")
check("i9 action kept unchanged", r.get("issue_action"), "labeled")
check("i9 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: issue_action")

r = one_silver("i10")
check("i10 action", r.get("issue_action"), None)
check("i10 warning", r.get("quarantine_warning"), "MISSING: issue_action")

r = one_silver("i11")
check("i11 times", times("i11"), (None, "2024-02-01 09:00:00"))
check("i11 warning", r.get("quarantine_warning"), "INVALID_CAST: issue_created_at")

r = one_silver("i14")
check("i14 winner = newer ingestion", (r.get("issue_id"), r.get("_source_file")), (1402, F2))

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["JSON_PARSE_ERROR", "MISSING: repo_id"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("i12 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["i12"])
i13 = [r for r in q if r["event_id"] == "i13"]
check("i13 found once", len(i13), 1)
r = i13[0] if i13 else {}
check("i13 issue fields kept", (r.get("issue_action"), r.get("issue_id"), r.get("issue_number")),
      ("opened", 7001, 11))
check("i13 original text kept",
      (r.get("issue_id_raw"), r.get("issue_number_raw"),
       r.get("issue_created_at_raw"), r.get("issue_closed_at_raw")),
      ("7001", "11", "2024-01-15T08:00:00Z", None))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../03_issue_events $test_schema="test_issue" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 15, "silver_candidate_rows": 12,
    "silver_rows_inserted": 0, "silver_rows_already_present": 12,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 12)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")